# Kubernetes Assignments — PW Skills
**Course:** Java + DSA — PW Skills  
**Topic:** Kubernetes (Deployments, Services, Scaling, Ingress)

This notebook covers all **five Kubernetes assignments**:

| # | Task |
|---|---|
| 1 | Deploy a 3-node Kubernetes cluster + NGINX deployment with 3 replicas |
| 2 | Add a **NodePort** service for NGINX, verify in a browser |
| 3 | Scale the NGINX deployment to **5 replicas** |
| 4 | Change the service type from NodePort to **ClusterIP** |
| 5 | Add an **Apache** deployment + an **Ingress** routing `/nginx` → NGINX, `/apache` → Apache |



## 🔧 Setup — Installing minikube + kubectl

A real 3-node cluster on cloud VMs would cost money. **minikube** spins up a multi-node cluster on your laptop for free and is what this assignment expects. Pick the instructions for your OS.

### macOS (Homebrew)
```bash
brew install minikube kubectl
```

### Windows (Chocolatey)
```powershell
choco install minikube kubernetes-cli
```

### Linux (Debian/Ubuntu)
```bash
# kubectl
curl -LO "https://dl.k8s.io/release/$(curl -L -s https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
sudo install -o root -g root -m 0755 kubectl /usr/local/bin/kubectl

# minikube
curl -LO https://storage.googleapis.com/minikube/releases/latest/minikube-linux-amd64
sudo install minikube-linux-amd64 /usr/local/bin/minikube
```

You also need a container/VM driver — **Docker Desktop** is the easiest option on all three OSes.

### Verify the install
```bash
minikube version
kubectl version --client
docker --version
```

---

## 🟦 Assignment 1 — 3-node cluster + NGINX deployment (3 replicas)

### Tasks
1. Deploy a Kubernetes cluster with **3 nodes**
2. Create an **NGINX deployment with 3 replicas**

### Step 1.1 — Start a 3-node minikube cluster

```bash
minikube start --nodes 3 --driver=docker
```

This takes 2–4 minutes the first time. minikube downloads the Kubernetes images, creates 3 Docker containers (one control-plane node + two workers), and wires them together.

### Step 1.2 — Confirm 3 nodes are Ready

```bash
kubectl get nodes -o wide
```

**Expected output (yours will show different IPs/timestamps):**
```
NAME           STATUS   ROLES           AGE     VERSION   INTERNAL-IP    OS-IMAGE
minikube       Ready    control-plane   3m17s   v1.30.0   192.168.49.2   Ubuntu 22.04.4 LTS
minikube-m02   Ready    <none>          2m48s   v1.30.0   192.168.49.3   Ubuntu 22.04.4 LTS
minikube-m03   Ready    <none>          2m25s   v1.30.0   192.168.49.4   Ubuntu 22.04.4 LTS
```


### Step 1.3 — Create the NGINX deployment manifest

Save this as **`nginx-deployment.yaml`**:

In [ ]:
# nginx-deployment.yaml
%%writefile nginx-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx-deployment
  labels:
    app: nginx
spec:
  replicas: 3                          # ← Assignment 1 asks for 3
  selector:
    matchLabels:
      app: nginx
  template:
    metadata:
      labels:
        app: nginx
    spec:
      containers:
      - name: nginx
        image: nginx:1.27
        ports:
        - containerPort: 80


### Step 1.4 — Apply and verify

```bash
kubectl apply -f nginx-deployment.yaml
kubectl get deployments
kubectl get pods -o wide
```

**Expected:**
```
deployment.apps/nginx-deployment created

NAME               READY   UP-TO-DATE   AVAILABLE   AGE
nginx-deployment   3/3     3            3           45s

NAME                                READY   STATUS    RESTARTS   AGE   IP            NODE
nginx-deployment-7c8b56f9c8-2tnpw   1/1     Running   0          45s   10.244.1.2    minikube-m02
nginx-deployment-7c8b56f9c8-4kgxd   1/1     Running   0          45s   10.244.2.2    minikube-m03
nginx-deployment-7c8b56f9c8-jdpqz   1/1     Running   0          45s   10.244.0.5    minikube
```

Notice how the 3 pods get scheduled across the 3 different nodes — that's the scheduler doing its job.



## 🟩 Assignment 2 — NodePort service for NGINX

### Tasks
1. Use the deployment from Assignment 1
2. Create a **NodePort** service for the NGINX deployment
3. Verify it in a browser

### Step 2.1 — What's a NodePort service?

A **NodePort** service exposes the deployment on a fixed port (30000–32767) on **every node in the cluster**. Hitting any node IP on that port reaches one of the NGINX pods (kube-proxy load-balances).

### Step 2.2 — Create the service manifest

Save as **`nginx-nodeport.yaml`**:

In [ ]:
# nginx-nodeport.yaml
%%writefile nginx-nodeport.yaml
apiVersion: v1
kind: Service
metadata:
  name: nginx-service
spec:
  type: NodePort                       # ← Key bit for Assignment 2
  selector:
    app: nginx                         # ← Matches the deployment's pod labels
  ports:
  - port: 80                           # cluster-internal port
    targetPort: 80                     # port the container listens on
    nodePort: 30080                    # external port (must be 30000-32767)


### Step 2.3 — Apply and verify

```bash
kubectl apply -f nginx-nodeport.yaml
kubectl get svc
```

**Expected:**
```
service/nginx-service created

NAME            TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)        AGE
kubernetes      ClusterIP   10.96.0.1       <none>        443/TCP        12m
nginx-service   NodePort    10.107.42.131   <none>        80:30080/TCP   5s
```

The `80:30080/TCP` notation means: cluster-internal port 80, exposed on every node at port 30080.

### Step 2.4 — Open in a browser

minikube makes this easy:
```bash
minikube service nginx-service --url
```

This prints a URL like `http://192.168.49.2:30080`. Open it in your browser — you'll see the **NGINX welcome page** ("Welcome to nginx!").



## 🟨 Assignment 3 — Scale to 5 replicas

### Tasks
1. Use the previous deployment
2. Change the replica count to **5**

### Step 3.1 — Two ways to do it

**Option A (quick, imperative):**
```bash
kubectl scale deployment nginx-deployment --replicas=5
```

**Option B (the "GitOps way" — edit the YAML and re-apply):**
```yaml
# nginx-deployment.yaml — only the changed line
spec:
  replicas: 5         # ← was 3
```
```bash
kubectl apply -f nginx-deployment.yaml
```

Option B is what you'd do in a real codebase — the manifest in git is the source of truth. For this assignment either is fine; show whichever one you used.

### Step 3.2 — Verify

```bash
kubectl get deployment nginx-deployment
kubectl get pods -o wide
```

**Expected after 10–20 seconds:**
```
NAME               READY   UP-TO-DATE   AVAILABLE   AGE
nginx-deployment   5/5     5            5           8m

NAME                                READY   STATUS    RESTARTS   AGE
nginx-deployment-7c8b56f9c8-2tnpw   1/1     Running   0          8m
nginx-deployment-7c8b56f9c8-4kgxd   1/1     Running   0          8m
nginx-deployment-7c8b56f9c8-jdpqz   1/1     Running   0          8m
nginx-deployment-7c8b56f9c8-mr8x2   1/1     Running   0          22s    ← new
nginx-deployment-7c8b56f9c8-q9wls   1/1     Running   0          22s    ← new
```

Notice the AGE column — three pods are old, two are brand new. That's Kubernetes adding only the pods it needs, without disturbing the existing ones. This is what makes scaling "free" — no downtime.



## 🟧 Assignment 4 — Change service type to ClusterIP

### Tasks
1. Use the previous deployment
2. Change the service type from NodePort to **ClusterIP**

### Step 4.1 — What's the difference?

| Type | Reachable from |
|---|---|
| **NodePort** | Outside the cluster, via any node's IP on a fixed port |
| **ClusterIP** | **Only from inside the cluster** (other pods/services) — the default |

ClusterIP is what you want for **internal services** (database, cache, internal APIs). Anything user-facing typically goes through an Ingress (Assignment 5) which itself talks to ClusterIP services internally.

### Step 4.2 — Update the service manifest

Edit **`nginx-nodeport.yaml`** (or rename it to `nginx-clusterip.yaml` to be cleaner):

In [ ]:
# nginx-clusterip.yaml
%%writefile nginx-clusterip.yaml
apiVersion: v1
kind: Service
metadata:
  name: nginx-service
spec:
  type: ClusterIP                      # ← Changed from NodePort
  selector:
    app: nginx
  ports:
  - port: 80
    targetPort: 80
                                       # No nodePort field anymore


### Step 4.3 — Apply

```bash
# Delete the old NodePort service first, since you can't switch type
# from NodePort to ClusterIP with `apply` directly in some k8s versions
kubectl delete svc nginx-service
kubectl apply -f nginx-clusterip.yaml
kubectl get svc
```

**Expected:**
```
service "nginx-service" deleted
service/nginx-service created

NAME            TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)   AGE
kubernetes      ClusterIP   10.96.0.1       <none>        443/TCP   25m
nginx-service   ClusterIP   10.103.55.207   <none>        80/TCP    3s
```

The `PORT(S)` column now shows just `80/TCP` — no external port. The browser URL from Assignment 2 no longer works.

### Step 4.4 — Prove it still works from inside the cluster

```bash
kubectl run tmp-shell --rm -it --image=busybox -- sh
# Inside the pod:
wget -qO- http://nginx-service:80
# Should print NGINX's HTML welcome page
exit
```



## 🟪 Assignment 5 — Apache deployment + Ingress routing

### Tasks
1. Use the previous deployment (NGINX deployment + ClusterIP service still there)
2. Deploy an **Apache** deployment with 3 replicas
3. Create an Apache **ClusterIP** service
4. Create an **Ingress** that routes `/nginx` → NGINX service, `/apache` → Apache service

### Step 5.1 — Enable the ingress addon

minikube ships with the NGINX Ingress Controller as an optional addon:
```bash
minikube addons enable ingress
kubectl get pods -n ingress-nginx
```

Wait until the `ingress-nginx-controller-...` pod is `Running`. Takes 30–60 seconds.

### Step 5.2 — Apache deployment + ClusterIP service

Save as **`apache.yaml`**:

In [ ]:
# apache.yaml — deployment + service in one file
%%writefile apache.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: apache-deployment
  labels:
    app: apache
spec:
  replicas: 3
  selector:
    matchLabels:
      app: apache
  template:
    metadata:
      labels:
        app: apache
    spec:
      containers:
      - name: apache
        image: httpd:2.4
        ports:
        - containerPort: 80
---
apiVersion: v1
kind: Service
metadata:
  name: apache-service
spec:
  type: ClusterIP
  selector:
    app: apache
  ports:
  - port: 80
    targetPort: 80


```bash
kubectl apply -f apache.yaml
kubectl get deploy,svc,pods -l app=apache
```

**Expected:**
```
deployment.apps/apache-deployment created
service/apache-service created

NAME                                READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/apache-deployment   3/3     3            3           20s

NAME                     TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)   AGE
service/apache-service   ClusterIP   10.100.23.115   <none>        80/TCP    20s

NAME                                     READY   STATUS    RESTARTS   AGE
pod/apache-deployment-6cbf7b8c54-7vfdq   1/1     Running   0          20s
pod/apache-deployment-6cbf7b8c54-h2k9p   1/1     Running   0          20s
pod/apache-deployment-6cbf7b8c54-x8mlc   1/1     Running   0          20s
```



### Step 5.3 — The Ingress

Save as **`ingress.yaml`**:

In [ ]:
# ingress.yaml — path-based routing to two services
%%writefile ingress.yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: web-ingress
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /
spec:
  ingressClassName: nginx
  rules:
  - http:
      paths:
      - path: /nginx
        pathType: Prefix
        backend:
          service:
            name: nginx-service        # from Assignment 4 (ClusterIP)
            port:
              number: 80
      - path: /apache
        pathType: Prefix
        backend:
          service:
            name: apache-service       # from Step 5.2
            port:
              number: 80


```bash
kubectl apply -f ingress.yaml
kubectl get ingress
```

**Expected (the ADDRESS field may take 30–60 seconds to populate):**
```
NAME          CLASS   HOSTS   ADDRESS        PORTS   AGE
web-ingress   nginx   *       192.168.49.2   80      45s
```

### Step 5.4 — Test the routing

```bash
# Get the ingress IP (or use `minikube ip`)
INGRESS_IP=$(minikube ip)
echo $INGRESS_IP

# Hit the NGINX route
curl http://$INGRESS_IP/nginx
# → returns NGINX welcome HTML (<title>Welcome to nginx!</title>)

# Hit the Apache route
curl http://$INGRESS_IP/apache
# → returns Apache welcome HTML (<html><body><h1>It works!</h1></body></html>)
```

**You can also test in a browser:**
- `http://<minikube-ip>/nginx`  → NGINX welcome page
- `http://<minikube-ip>/apache` → Apache "It works!" page


---

## 🧹 Cleanup (when you're done)

```bash
# Delete everything you created
kubectl delete ingress web-ingress
kubectl delete svc nginx-service apache-service
kubectl delete deployment nginx-deployment apache-deployment

# Stop the cluster
minikube stop

# Or wipe it completely
minikube delete
```



## 💡 Notes & Observations

A few things worth noticing as you work through this:

1. **Pods get scheduled across all 3 nodes.** Look at the NODE column in `kubectl get pods -o wide` — that's the scheduler making sure your workload survives a single-node failure.

2. **Scaling is fast and non-disruptive.** When you went from 3 to 5 replicas, the original 3 pods stayed running. Kubernetes only added what was missing.

3. **Switching service types from NodePort to ClusterIP "breaks" external access** — that's the *point*. ClusterIP is an internal-only service. The Ingress in Assignment 5 puts an external-facing layer back in place, but routes traffic to ClusterIP services internally. That's the production pattern: app services are ClusterIP, ingress is the front door.

4. **The Ingress is just a smart load balancer.** Two paths, two services, but it all looks like one address to the user. This is how you'd host multiple apps on a single domain.